### Data analysis (BSL3)
Reads all CSVs from `results/experiment_id`, concatenates them, saves under `processed_results/experiment_id`, computes mean features per well/timepoint, and plots plate views.

Filename parsing is BSL3-specific (e.g. `..._WellB02_timepoint4h_Pos013`, `..._WellB03_timepoint3d_Pos014`): extracts `well_id`, `timepoint` (days converted to hours), and `position`.

### Imports

In [ ]:
import os
from pathlib import Path

import pandas as pd
from utils_data_analysis import plot_plate_view


def extract_well_id_timepoint_and_position(df, filename_col="filename"):
    """
    Parse well_id, timepoint (hours), and position from BSL3 filenames.

    Expected patterns:
    - ``..._WellB02_timepoint4h_Pos013``
    - ``..._WellB03_timepoint3d_Pos014``

    Days (``d``) are converted to hours (×24) before storing ``timepoint``.
    Columns are inserted immediately after ``filename_col``.
    """
    out = df.copy()
    filenames = out[filename_col].astype(str)

    well_ids = filenames.str.extract(r"Well([A-Za-z]\d{1,2})_", expand=False)

    tp = filenames.str.extract(r"timepoint(\d+)([hd])_", expand=True)
    tp.columns = ["value", "unit"]
    hours = pd.to_numeric(tp["value"], errors="coerce")
    hours = hours.where(tp["unit"] != "d", hours * 24).astype("Int64")

    positions = filenames.str.extract(r"Pos(\d+)", expand=False)
    positions = pd.to_numeric(positions, errors="coerce").astype("Int64")

    insert_at = out.columns.get_loc(filename_col) + 1
    out.insert(insert_at, "well_id", well_ids)
    out.insert(insert_at + 1, "timepoint", hours)
    out.insert(insert_at + 2, "position", positions)
    return out

### Config

In [ ]:
# Experiment ID (must match the folder name under results/)
experiment_id = "maxprojections"

results_dir = Path("results") / experiment_id
processed_dir = Path("processed_results") / experiment_id
os.makedirs(processed_dir, exist_ok=True)
print(f"Reading from {results_dir}")
print(f"Writing to {processed_dir}")

### Read all CSVs, concatenate, and save

In [ ]:
csv_files = sorted([f for f in results_dir.glob("*.csv") if f.name != "infection_summary.csv"])
if not csv_files:
    raise FileNotFoundError(f"No CSV files in {results_dir}")

df_all = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

df_all = extract_well_id_timepoint_and_position(df_all)

out_path = processed_dir / "concatenated.csv"
df_all.to_csv(out_path, index=False)
print(f"Concatenated {len(csv_files)} files → {len(df_all)} rows. Saved to {out_path}")

In [ ]:
df_all

### Mean features per well and plate view of  (all cells)

In [ ]:
# Average numeric features by well_id and timepoint (exclude cell id, infection flag, FOV index)
cols_to_exclude = ["label", "Mtb_infected_cell", "position"]
df_for_mean = df_all.drop(columns=[c for c in cols_to_exclude if c in df_all.columns])
df_mean = df_for_mean.groupby(["well_id", "timepoint"], as_index=False).mean(numeric_only=True)

# Save mean dataframe
mean_path = processed_dir / "mean_per_well.csv"
df_mean.to_csv(mean_path, index=False)
print(f"Mean per well saved to {mean_path}")

In [ ]:
# Create a directory for plate view plots if doesn't exist
plate_view_dir = processed_dir / "plate_view"
plate_view_dir.mkdir(exist_ok=True)

for timepoint in df_mean["timepoint"].unique():
    df_time = df_mean[df_mean["timepoint"] == timepoint]
    for feature in df_mean.columns:
        if feature in ["well_id", "label", "timepoint", "position"]:
            continue
        # Create filename append for this timepoint
        timepoint_str = f"{int(timepoint)}h"
        plot_plate_view(
            df_time.copy(),
            column_name=feature,
            title=f"{feature} ({timepoint_str})",
            label=feature,
            save_dir=str(plate_view_dir),
            fmt=0,
            display=False,
            save_name=f"{feature}_{timepoint_str}.png",
        )

### Mean features per well and plate view of feature (only infected cells)

In [ ]:
# Average numeric features by well_id and timepoint for infected cells only
df_infected = df_all[df_all["Mtb_infected_cell"] == True]
cols_to_exclude = ["label", "Mtb_infected_cell", "position"]
df_for_mean_inf = df_infected.drop(columns=[c for c in cols_to_exclude if c in df_infected.columns])
df_mean_inf = df_for_mean_inf.groupby(["well_id", "timepoint"], as_index=False).mean(numeric_only=True)

# Save mean dataframe
mean_inf_path = processed_dir / "mean_per_well_only_inf_cells.csv"
df_mean_inf.to_csv(mean_inf_path, index=False)
print(f"Mean per well (infected cells only) saved to {mean_inf_path}")

In [ ]:
# Create a directory for plate view plots if doesn't exist
plate_view_dir = processed_dir / "plate_view"
plate_view_dir.mkdir(exist_ok=True)

for timepoint in df_mean_inf["timepoint"].unique():
    df_time = df_mean_inf[df_mean_inf["timepoint"] == timepoint]
    for feature in df_mean_inf.columns:
        if feature in ["well_id", "label", "timepoint", "position"]:
            continue
        # Create filename append for this timepoint
        timepoint_str = f"{int(timepoint)}h"
        plot_plate_view(
            df_time.copy(),
            column_name=feature,
            title=f"{feature} ({timepoint_str}, infected cells)",
            label=feature,
            save_dir=str(plate_view_dir),
            fmt=0,
            display=False,
            save_name=f"{feature}_{timepoint_str}_inf_cells.png",
        )

### Plate view of % infected cells (infection_summary)

In [ ]:
infection_summary_path = results_dir / "infection_summary.csv"
df_inf = pd.read_csv(infection_summary_path)
df_inf = extract_well_id_timepoint_and_position(df_inf)

plate_view_dir = processed_dir / "plate_view"
plate_view_dir.mkdir(exist_ok=True)

for timepoint in sorted(df_inf["timepoint"].dropna().unique()):
    df_time = df_inf[df_inf["timepoint"] == timepoint]
    timepoint_str = f"{int(timepoint)}h"
    plot_plate_view(
        df_time.copy(),
        column_name="%_inf_cells",
        title=f"% infected cells ({timepoint_str})",
        label="%_inf_cells",
        save_dir=str(plate_view_dir),
        fmt=1,
        display=False,
        save_name=f"%_inf_cells_{timepoint_str}.png",
    )